# Refined End-to-End Preferred API Workflow

This notebook mirrors the robust refined end-to-end smoke test using the newer preferred vector-builder API.

It writes real geopackage inputs on the fly, builds an optimized refined `TriangleGrid -> VoronoiGridPlus` mesh, and then assembles a single two-period MF6 model using:

- `CHDFromVector`
- `GHBFromVector`
- `DRNFromVector`
- `RCHFromVector`
- `KFromVector`
- `UZFBuilder`
- `LAK`
- `SFR`
- `MVRBuilder`

The surface-water coupling is deliberately mild so the single refined model stays stable while still attaching `SFR`, `LAK`, and `MVRBuilder` on the same grid.

In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import LineString, Point, box

import myflopy as mf
from myflopy.modflow.mf6.chd import CHDFromVector
from myflopy.modflow.mf6.drn import DRNFromVector
from myflopy.modflow.mf6.ghb import GHBFromVector
from myflopy.modflow.mf6.kflow import KFromVector
from myflopy.modflow.mf6.lakes import LAKBuilder
from myflopy.modflow.mf6.mvr import MVRBuilder, Move
from myflopy.modflow.mf6.recharge import RCHFromVector
from myflopy import SFRBuilder
from myflopy import ModelContext, UZFBuilder
from myflopy.modflow.mf6.simulation.discretization import DisvGrid, TemporalDiscretization
from myflopy.modflow.mf6.simulation.packages import (
    CHD,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)


In [ ]:
if Path.cwd().name == "notebooks" and Path.cwd().parent.name == "mf6":
    examples_root = Path.cwd().parent
else:
    examples_root = Path.cwd() / "examples" / "mf6"

workspace = examples_root / "artifacts" / "refined_end_to_end_preferred_api_workflow"
if workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir(parents=True, exist_ok=True)

workspace

In [ ]:
def write_gpkg(path: Path, gdf: gpd.GeoDataFrame) -> Path:
    gdf.to_file(path, driver="GPKG")
    return path


def write_refined_workflow_vectors(workspace: Path) -> dict[str, object]:
    crs = "EPSG:2927"
    domain_geom = box(0.0, 0.0, 5000.0, 4000.0)
    west_band = box(0.0, 0.0, 225.0, 4000.0)
    east_band = box(4775.0, 0.0, 5000.0, 4000.0)
    north_band = box(0.0, 3650.0, 5000.0, 4000.0)

    stream_main = LineString([
        (250.0, 3500.0), (1200.0, 3050.0), (2300.0, 2400.0), (3550.0, 1650.0), (4750.0, 850.0)
    ])
    stream_branch = LineString([
        (650.0, 950.0), (1500.0, 1400.0), (2500.0, 1900.0), (3650.0, 2250.0), (4550.0, 2500.0)
    ])
    lake_west = Point(1650.0, 2550.0).buffer(290.0, quad_segs=24)
    lake_east = Point(3350.0, 1500.0).buffer(250.0, quad_segs=24)
    recharge_west = box(350.0, 2200.0, 2200.0, 3800.0)
    recharge_central = box(1850.0, 900.0, 3400.0, 2500.0)
    recharge_east = box(3100.0, 1300.0, 4700.0, 3400.0)
    k_west = box(0.0, 0.0, 2100.0, 4000.0)
    k_central = box(1650.0, 700.0, 3450.0, 3200.0)
    k_east = box(2900.0, 0.0, 5000.0, 4000.0)

    domain_path = write_gpkg(workspace / "refined_domain.gpkg", gpd.GeoDataFrame({"name": ["domain"]}, geometry=[domain_geom], crs=crs))
    stream_paths = [
        write_gpkg(workspace / "stream_main.gpkg", gpd.GeoDataFrame({"name": ["stream_main"]}, geometry=[stream_main], crs=crs)),
        write_gpkg(workspace / "stream_branch.gpkg", gpd.GeoDataFrame({"name": ["stream_branch"]}, geometry=[stream_branch], crs=crs)),
    ]
    lake_paths = [
        write_gpkg(workspace / "lake_west.gpkg", gpd.GeoDataFrame({"name": ["lake_west"]}, geometry=[lake_west], crs=crs)),
        write_gpkg(workspace / "lake_east.gpkg", gpd.GeoDataFrame({"name": ["lake_east"]}, geometry=[lake_east], crs=crs)),
    ]
    chd_path = write_gpkg(workspace / "boundary_chd.gpkg", gpd.GeoDataFrame({"name": ["west_chd"], "elev": [0.0], "layer": [1]}, geometry=[west_band], crs=crs))
    ghb_path = write_gpkg(
        workspace / "boundary_ghb.gpkg",
        gpd.GeoDataFrame({"name": ["east_ghb"], "elev": [0.0], "height": [0.0], "cond": [20.0], "layer": [1], "min_elev": [0.0]}, geometry=[east_band], crs=crs),
    )
    drn_path = write_gpkg(
        workspace / "boundary_drn.gpkg",
        gpd.GeoDataFrame({"name": ["north_drn"], "height": [-1.0], "cond": [5.0], "layer": [1], "min_elev": [0.0]}, geometry=[north_band], crs=crs),
    )
    rch_path = write_gpkg(
        workspace / "recharge_zones.gpkg",
        gpd.GeoDataFrame(
            {
                "zone": ["west_uplands", "central_infiltration", "east_uplands"],
                "rch_0": [0.00005, 0.00008, 0.00004],
                "rch_1": [0.00008, 0.00012, 0.00006],
            },
            geometry=[recharge_west, recharge_central, recharge_east],
            crs=crs,
        ),
    )
    k_path = write_gpkg(
        workspace / "k_zones.gpkg",
        gpd.GeoDataFrame({"name": ["west_k", "central_k", "east_k"], "k": [22.0, 12.0, 16.0], "layer": [1, 1, 1]}, geometry=[k_west, k_central, k_east], crs=crs),
    )
    refinement_paths = [
        write_gpkg(workspace / "refine_recharge_west.gpkg", gpd.GeoDataFrame({"name": ["west_uplands"]}, geometry=[recharge_west], crs=crs)),
        write_gpkg(workspace / "refine_recharge_central.gpkg", gpd.GeoDataFrame({"name": ["central_infiltration"]}, geometry=[recharge_central], crs=crs)),
        write_gpkg(workspace / "refine_recharge_east.gpkg", gpd.GeoDataFrame({"name": ["east_uplands"]}, geometry=[recharge_east], crs=crs)),
    ]
    return {
        "crs": crs,
        "domain": domain_path,
        "streams": stream_paths,
        "lakes": lake_paths,
        "chd": chd_path,
        "ghb": ghb_path,
        "drn": drn_path,
        "rch": rch_path,
        "k": k_path,
        "refinement_regions": refinement_paths,
    }


def build_refined_triangle(workspace: Path):
    inputs = write_refined_workflow_vectors(workspace)
    tri = mf.TriangleGrid(model_ws=str(workspace / "refined_mesh"), angle=30)
    tri.set_domain_file(inputs["domain"], simplify_tolerance=5, densify_dist=125, max_area=20000, label="domain")
    for stream_path, label in zip(inputs["streams"], ["stream_main", "stream_branch"], strict=True):
        tri.add_line_feature(stream_path, buffer=55, simplify_tolerance=5, densify_dist=75, max_area=3000, label=label, priority=5)
    for lake_path, label in zip(inputs["lakes"], ["lake_west", "lake_east"], strict=True):
        tri.add_region_file(lake_path, simplify_tolerance=5, densify_dist=50, max_area=2500, label=label, priority=6)
    for idx, refinement_path in enumerate(inputs["refinement_regions"]):
        tri.add_region_file(refinement_path, simplify_tolerance=5, densify_dist=80, max_area=5500, label=f"refine_zone_{idx}", priority=4)
    tri.add_region_polygon(box(900.0, 900.0, 4100.0, 3200.0), densify_dist=100, max_area=10000, label="interior_refinement", priority=3)
    mesh_report = tri.build_mesh(
        profile="balanced",
        protect_sources=("line",),
        protected_labels=["stream_main", "stream_branch", "lake_west", "lake_east"],
        target_segment_length=100,
        optimization_iterations=1,
        verbose=False,
    )
    vor = mf.VoronoiGridPlus(tri, crs=inputs["crs"], name="refined_vector_test")
    return tri, vor, mesh_report, inputs


tri, vor, mesh_report, inputs = build_refined_triangle(workspace)
vor.ncpl

In [ ]:
tri.preview_regions()[["label", "max_area", "claim_area", "priority"]]

In [ ]:
vor.plot.grid()

In [ ]:
centroids = vor.gdf_vorPolys.geometry.centroid
top = 142.0 - (centroids.x.to_numpy() / 150.0) - (centroids.y.to_numpy() / 500.0)
bottom = top - 75.0 - 4.0 * np.sin(centroids.x.to_numpy() / 850.0)
bottom = np.minimum(bottom, top - 5.0)
vor.gdf_topbtm = gpd.GeoDataFrame(
    {"geometry": vor.gdf_vorPolys.geometry, 0: top, 1: bottom},
    geometry="geometry",
    crs=vor.crs,
)

top_arr = np.asarray(top, dtype=float)
bottom_arr = np.asarray(bottom, dtype=float)
initial_heads = np.maximum(bottom_arr + 10.0, top_arr - 8.0).tolist()

model = mf.SimulationBase(name="refined_e2e", mf_folder_path=workspace, vor=vor, nper=2)
DisvGrid(vor=vor, model=model, top=top.tolist(), bottom=[bottom.tolist()], nlay=1, idomain=[[1] * vor.ncpl])
TemporalDiscretization(model=model, period_data=[[1.0, 1, 1.0], [1.0, 1, 1.0]])
InitialConditions(model=model, vor=vor, nlay=1, strt=initial_heads)

k_builder = KFromVector(model=model, vor=vor, shp_gpkg=inputs["k"], uid="name")
k_array = k_builder.from_vector(defaults=[18.0])
KFlow(model=model, k=k_array[0].tolist(), k33_vert=(k_array[0] * 0.2).tolist(), save_specific_discharge=False)
Storage(model=model, sto_steady={0: True}, sto_transient={1: True})
OutputControl(model=model)

west_stage = float(np.nanpercentile(top_arr, 92) - 5.0)
east_stage = west_stage - 20.0

chd_builder = CHDFromVector(model=model, vor=vor, shp_gpkg=inputs["chd"], uid="name")
chd_dict = chd_builder.from_vector(head_reference={"west_chd": [west_stage, west_stage]}, register_regions=True, region_name_prefix="chd_group", combined_region_name="all_chd", overwrite_regions=True)
CHD(model=model, stress_period_data=chd_dict)

ghb_builder = GHBFromVector(model=model, vor=vor, shp_gpkg=inputs["ghb"], uid="name")
ghb_dict = ghb_builder.from_vector(elev_reference={"east_ghb": [east_stage, east_stage - 0.5]}, register_regions=True, region_name_prefix="ghb_group", combined_region_name="all_ghb", overwrite_regions=True)
mf.modflow.mf6.simulation.packages.GHB(model=model, stress_period_data=ghb_dict)

drn_builder = DRNFromVector(model=model, vor=vor, shp_gpkg=inputs["drn"], uid="name")
drn_dict = drn_builder.from_vector(edges_only=False, top_drain=True, register_regions=True, region_name_prefix="drn_group", combined_region_name="all_drains", overwrite_regions=True)
mf.modflow.mf6.simulation.packages.Drains(model=model, stress_period_data=drn_dict)

recharge_builder = RCHFromVector(model=model, vor=vor, shp_gpkg=inputs["rch"], uid="zone", rch_fields=["rch_0", "rch_1"], rch_fields_to_pers=[0, 1], background_rch=0.0, grid_type="disv", limit_to_k33=False)
rch_dict = recharge_builder.from_vector(register_regions=True, region_name_prefix="rch_zone", combined_region_name="all_rch", overwrite_regions=True)
Recharge(model=model, vor=vor, rch_dict=rch_dict)

uzf_context = ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain))
uzf_cells = [(0, cell) for cell in range(vor.ncpl)]
uzf_finf = {period: [{tuple(row[0]): row[1] for row in rows}.get(cellid, 0.0) for cellid in uzf_cells] for period, rows in rch_dict.items()}
uzf = UZFBuilder(context=uzf_context, nper=model.nper, cells=uzf_cells, vks=0.05, thtr=0.08, thts=0.28, thti=0.18, finf=uzf_finf)
uzf.build().build(model.gwf)
model.add_region_from_cells("uzf_all", uzf.uzf_cells, category="boundary", package="uzf", overwrite=True)

lak = LAKBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    lakes=[inputs["lakes"][0]],
    lake_id_field="name",
    starting_stage=129.0,
    lake_bottom=118.0,
    bed_leakance=0.001,
    connection_modes="automatic",
    status="ACTIVE",
    mover=True,
)
lak.build().build(model.gwf)
for lake_id, cells in lak.lake_cells.items():
    model.add_region_from_cells(f"lake_zone_{lake_id}", [(0, cell) for cell in cells], category="boundary", package="lak", tags=["lak"], geometry=lak.lake_table.loc[lake_id].geometry, overwrite=True)
model.add_region_from_cells("all_lakes", [(0, cell) for cells in lak.lake_cells.values() for cell in cells], category="boundary", package="lak", tags=["lak"], overwrite=True)

sfr = SFRBuilder(context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)), nper=model.nper, streams=[inputs["streams"][0]], inflow={0: [(0, 0.005)], 1: [(0, 0.005)]}, width=5.0, gradient=0.0008, roughness=0.03, streambed_k=0.05, streambed_thickness=2.0, mover=True)
sfr.build().build(model.gwf)
model.add_region_from_cells("all_streams", [(0, cell) for cells in sfr.stream_cells.values() for cell in cells], category="boundary", package="sfr", overwrite=True)

mvr = MVRBuilder(
    nper=model.nper,
    moves={
        period: [Move(source=sfr.connection(sfr.stream_ids[0]), receiver=lak.connection(lak.lake_ids[0]), value=0.25)]
        for period in range(model.nper)
    },
)
validation_report = model.validate_surface_water(
    nlakes=1,
    lak_packagedata=lak.packagedata,
    lak_connectiondata=lak.connectiondata,
    lak_perioddata=lak.perioddata,
    sfr=sfr,
    maxmvr=1,
    maxpackages=2,
    mvr_packages=mvr.packages,
    mvr_perioddata=mvr.perioddata,
    raise_on_error=True,
)

mvr.build().build(model.gwf)

vor.ncpl, len(connectiondata), sfr.total_nreaches

In [ ]:
validation_report.summary_frame()

In [ ]:
validation_report.to_frame()

In [ ]:
"sfr" in model.gwf.package_name_dict, "mvr" in model.gwf.package_name_dict, sorted(model.gwf.package_name_dict.keys())

In [ ]:
success, buff = model.run_simulation()
success

In [ ]:
sorted(path.name for path in model.model_output_folder_path.glob("*.sfr*")), sorted(path.name for path in model.model_output_folder_path.glob("*.mvr*"))

In [ ]:
model.summary()

In [ ]:
model.list_regions().tail(12)

In [ ]:
lak_stage = model.outputs.lak.stage.get()
sfr_stage = model.outputs.sfr.stage.get()
lak_stage
sfr_stage

In [ ]:
heads = model.gwf.output.head().get_data(kstpkper=(0, 1)).squeeze()
pd.DataFrame({"cell": np.arange(vor.ncpl), "head": heads}).head(12)

In [ ]:
import flopy

flopy.plot.PlotUtilities._plot_bc_helper(package=model.gwf.drn, kper=0)

In [ ]:
model.outputs.uzf.ifno_to_cellid